# Faruq-v3 — visual label audit

Audit CPU-only ini menampilkan label ukuran dan pasangan cacat lokal secara berdampingan. Tidak ada training, inference, atau akses test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
import coffee_detector
os.chdir(REPO)
print('IMPORT:', coffee_detector.__file__)

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_diagnostic.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DIAGNOSTIC = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_diagnostic.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1/label_visual_audit'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT   :', PROJECT_ROOT)
print('DATA      :', DATA_ROOT)
print('DIAGNOSTIC:', DIAGNOSTIC)
print('OUTPUT    :', OUTPUT_ROOT)

In [ ]:
from coffee_detector.analysis.faruq_v3_label_visual_audit import audit_faruq_v3_label_visuals

result = audit_faruq_v3_label_visuals(
    DATA_ROOT, DIAGNOSTIC, OUTPUT_ROOT,
    samples_per_class=6, max_local_pairs=6,
)
assert result['training_executed'] is False
assert result['inference_executed'] is False
assert result['test_images_accessed'] is False
print('AUDIT SELESAI:', result['decision'])

In [ ]:
import pandas as pd
from IPython.display import Image as DisplayImage, display

size_table = pd.DataFrame([{
    'split': row['split'],
    'family': row['family'],
    **{f'n_{level}': count for level, count in row['available'].items()},
    'contact_sheet': row['contact_sheet'],
} for row in result['size_sheets']])
pair_table = pd.DataFrame([{
    'rank': row['rank'], 'split': row['split'],
    'expected': row['expected'], 'predicted': row['predicted'],
    'count': row['confusion_count'], 'contact_sheet': row['contact_sheet'],
} for row in result['local_pair_sheets']])
display(size_table)
display(pair_table)

for row in result['size_sheets']:
    print(f"SIZE: {row['split']} / {row['family']}")
    display(DisplayImage(filename=row['contact_sheet'], width=1400))
for row in result['local_pair_sheets']:
    print(f"PAIR: {row['split']} / {row['expected']} -> {row['predicted']}")
    display(DisplayImage(filename=row['contact_sheet'], width=1400))

print('DECISION :', result['decision'])
print('TRAINING :', result['training_executed'])
print('INFERENCE:', result['inference_executed'])
print('TEST     :', result['test_images_accessed'])
print('SUMMARY  :', result['summary'])
print('Kirim contact sheet kulit_tanduk train+val dan pasangan lokal yang tampak tidak konsisten. Jangan training model baru.')